# 04 – Full pipeline demo video
In this notebook, we render the final demo video. The pipeline detects
people, tracks them, estimates their pose, classifies their state, and draws
the triage overlay on a held-out Okutama test video. It uses the fine-tuned
weights from notebooks 01 and 03.


In [2]:
# Install the packages we need.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


CUDA available: True


Saving src.zip to src.zip
Mounted at /content/drive


In [3]:
# Download Okutama-Action files from the public Dropbox folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    """Download and unpack one Okutama archive unless it is already present."""
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [4]:
# Held-out footage the models have never seen.
fetch_okutama('TestSetVideos.zip')

import glob, subprocess

def find_videos():
    """Collect Okutama video paths case-insensitively, after unpacking any
    nested archives that the folder download may have produced."""
    for z in glob.glob('/content/data/okutama/**/*.zip', recursive=True):
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/content/data/okutama'],
                       check=False)
    vids = []
    for ext in ('mov', 'MOV', 'mp4', 'MP4'):
        vids += glob.glob(f'/content/data/okutama/**/*.{ext}', recursive=True)
    return sorted(set(vids))

test_videos = find_videos()
# Fall back to the smaller Sample archive if the test set did not unpack.
if not test_videos:
    fetch_okutama('Sample.zip')
    test_videos = find_videos()

assert test_videos, 'No Okutama videos were found; inspect the download output above.'
print(len(test_videos), 'videos found:', test_videos[:3])


10 videos found: ['/content/data/okutama/Drone1/Morning/1.1.8.mp4', '/content/data/okutama/Drone1/Morning/1.1.9.mp4', '/content/data/okutama/Drone1/Noon/1.2.1.mp4']


In [5]:
import config
config.MODELS_DIR = OUT   # Loads pose_mlp.pt, the classifier from notebook 03.
from render_demo import render
det_weights = str(OUT/'yolo11s_visdrone_best.pt')  # From notebook 01 (or 'yolo11s.pt').
out, n = render(test_videos[0], '/content/demo_raw.mp4', weights=det_weights,
                imgsz=1280, max_frames=1800, device=0, pose_device='cuda',
                caption='EECS 4422 - SAR triage pipeline (held-out footage)')


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip
100%|██████████| 48.4M/48.4M [00:02<00:00, 20.7MB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.onnx with onnxruntime backend
[demo] Pose backend = rtmpose, state model = rule-based
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 262ms
Prepared 1 package in 43ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  Frame 30: 6 mobile
  Frame 60: 5 mobile
  Frame 90: 1 motionless · 4 mobile
  Frame 120: 3 motionless · 2 mobile
  Frame 150: 3 motionless · 3 mobile
  Frame 180: 2 motionless · 1 mobile
  Frame 210: 3 motionless · 2 mobile
  Frame 240: 3 motionless · 2 mobile
  Frame 270: 1 motionless · 5 mobile
  Frame 300: 4 mobile
  Frame 330: 2 mobile
  Frame 360: 1 motionless · 1 mobile
  Frame 390: 1 motionless
  Frame 420: no people
  Frame 450: no people
 

In [6]:
# Re-encode with H.264 for broad playback, then store it in Drive.
!ffmpeg -y -loglevel error -i /content/demo_raw.mp4 -c:v libx264 -pix_fmt yuv420p {OUT}/demo_final.mp4
print('Saved to Drive:', OUT/'demo_final.mp4')


Saved to Drive: /content/drive/MyDrive/sar_project_results/demo_final.mp4
